# Skin Cancer Detection — HAM10000

## Step 1: Install Dependencies

In [ ]:
!pip install kaggle tensorflow scikit-learn matplotlib seaborn pillow -q

## Step 2: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout,
    BatchNormalization, Input
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import load_model
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

Libraries imported successfully!
TensorFlow version: 2.19.0


## Step 3: Load Dataset

In [ ]:
# =============================================================
#  DATASET SETUP — Please follow these steps before running
# =============================================================
#
#  1. Go to this link and download the dataset:
#     https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
#
#  2. Click the "Download" button (top right) — it will download
#     a ZIP file named: skin-cancer-mnist-ham10000.zip
#
#  3. Extract the ZIP file
#
#  4. Place the extracted folder inside a 'data' folder
#     in the SAME directory as this notebook, like this:
#
#     skin_cancer_detection.ipynb   ← this notebook
#     data/
#     └── skin-cancer-mnist-ham10000/
#         ├── HAM10000_metadata.csv
#         ├── HAM10000_images_part_1/
#         └── HAM10000_images_part_2/
#
#  5. Once done, run the cell below to verify the dataset is found
# =============================================================

import os
from pathlib import Path

dataset_path = Path('data/skin-cancer-mnist-ham10000')

if dataset_path.exists():
    print(f'✅ Dataset found at: {dataset_path}\n')
    print('Contents:')
    for item in sorted(dataset_path.iterdir()):
        kind = 'DIR ' if item.is_dir() else 'FILE'
        print(f'  [{kind}] {item.name}')
else:
    print('❌ Dataset NOT found!')
    print(f'   Expected at: {dataset_path.resolve()}')
    print('   Please follow the instructions above.')

Dataset found at: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000

Contents:
  [DIR ] HAM10000_images_part_1
  [DIR ] HAM10000_images_part_2
  [FILE] HAM10000_metadata.csv
  [DIR ] ham10000_images_part_1
  [DIR ] ham10000_images_part_2
  [FILE] hmnist_28_28_L.csv
  [FILE] hmnist_28_28_RGB.csv
  [FILE] hmnist_8_8_L.csv
  [FILE] hmnist_8_8_RGB.csv


## Step 4: Exploring Metadata

In [ ]:
df = pd.read_csv('data/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')

In [ ]:
print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

Dataset Shape: (10015, 7)

First 5 rows:


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [ ]:
print(f'Class Distribution:')
print(df['dx'].value_counts())
print(f'\nMissing values:')
print(df.isnull().sum())

Class Distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Missing values:
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64


In [ ]:
df['age'] = df['age'].fillna(df['age'].median())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index, palette='viridis')
plt.title('Class Distribution in HAM10000 Dataset', fontsize=14)
plt.xlabel('Skin Lesion Type')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## Step 5: Exploring Sample Images

In [ ]:
# Display sample images from each class
import os

folders = [
    'data/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
    'data/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
]

def find_image(image_id):
    for folder in folders:
        path = os.path.join(folder, image_id + '.jpg')
        if os.path.exists(path):
            return path
    return None

df['image_path'] = df['image_id'].apply(find_image)

# Check for any missing
missing = df['image_path'].isna().sum()
print(f"Found: {len(df) - missing} | Missing: {missing}")
class_names_full = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions'
}

fig, axes = plt.subplots(7, 5, figsize=(20, 28))
fig.suptitle('Sample Images from Each Skin Lesion Class', fontsize=18, y=1.01)

for row_idx, cls in enumerate(sorted(df['dx'].unique())):
    cls_df = df[df['dx'] == cls].sample(n=5, random_state=42)
    for col_idx, (_, sample) in enumerate(cls_df.iterrows()):
        img = Image.open(sample['image_path']).convert('RGB')
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_title(
                f"{cls.upper()} — {class_names_full.get(cls, cls)}\n(n={len(df[df['dx']==cls])})",
                fontsize=11, fontweight='bold', loc='left'
            )
        else:
            axes[row_idx, col_idx].set_title(f'{cls.upper()}', fontsize=10)

plt.tight_layout()
plt.savefig('sample_images_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Image Paths & Preprocessing

In [ ]:
IMG_SIZE = 224

df['age'].fillna(df['age'].median(), inplace=True)

le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])
classes = le.classes_
NUM_CLASSES = len(classes)

print(f'Records: {len(df)}')
print(f'Classes ({NUM_CLASSES}): {classes}')

Records: 10015
Classes (7): ['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']


In [ ]:
def load_and_preprocess_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert('RGB').resize((img_size, img_size))
    img = np.array(img, dtype=np.float32)
    img = preprocess_input(img)
    return img

print(f'Loading {len(df)} images at {IMG_SIZE}x{IMG_SIZE}...')
X = np.array([load_and_preprocess_image(p) for p in df['image_path']])
print(f'Image array shape: {X.shape}')

le = LabelEncoder()
y = le.fit_transform(df['dx'])
classes = le.classes_
NUM_CLASSES = len(classes)
print(f'Classes ({NUM_CLASSES}): {classes}')

Loading 10015 images at 224x224...
Image array shape: (10015, 224, 224, 3)
Classes (7): ['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']


### Visualizing Preprocessed Images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Preprocessed Images', fontsize=14)

sample_indices = np.random.RandomState(42).choice(len(X), 10, replace=False)

for i, idx in enumerate(sample_indices):
    row, col = i // 5, i % 5
    img_display = X[idx].copy()

    if img_display.max() > 1:
        img_display = img_display / 255.0
    elif img_display.min() < 0:
        img_display = (img_display + 1) / 2.0

    img_display = np.clip(img_display, 0, 1)
    axes[row, col].imshow(img_display)
    axes[row, col].set_title(f'Class: {classes[y[idx]]}', fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## Step 7: Train/Val/Test Split & Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples:       {len(X_test)}')

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_clipped = np.clip(class_weights_array, 0.5, 3.0)
class_weight_dict = dict(enumerate(class_weights_clipped))

print(f'\nOriginal balanced class weights vs. Clipped weights:')
for i, cls in enumerate(classes):
    print(f'  {cls}: {class_weights_array[i]:.3f} -> {class_weights_clipped[i]:.3f}')

y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat   = tf.keras.utils.to_categorical(y_val, NUM_CLASSES)
y_test_cat  = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

## Step 8: Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    shear_range=0.15,
    channel_shift_range=20.0,
    fill_mode='reflect'
)

datagen.fit(X_train)
print('Data augmentation configured.')

Data augmentation configured.


### Visualizing Augmented Images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def reverse_preprocess(img):

    img = img.copy()

    if img.max() > 1:
        img = img / 255.0
    elif img.min() < 0:
        img = (img + 1.0) / 2.0

    return np.clip(img, 0, 1)

fig, axes = plt.subplots(4, 6, figsize=(24, 16))
fig.suptitle('Original vs Augmented Images', fontsize=16, y=1.01)

sample_indices = np.random.RandomState(123).choice(len(X_train), 4, replace=False)

for row_idx, idx in enumerate(sample_indices):
    # Original
    original_img = reverse_preprocess(X_train[idx])
    axes[row_idx, 0].imshow(original_img)
    axes[row_idx, 0].set_title(f'Original\n({classes[y_train[idx]]})', fontsize=10, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Augmented versions
    img_batch = np.expand_dims(X_train[idx], 0)
    aug_iter = datagen.flow(img_batch, batch_size=1, seed=row_idx)

    for col_idx in range(1, 6):
        aug_img = next(aug_iter)[0]
        aug_img = reverse_preprocess(aug_img)

        axes[row_idx, col_idx].imshow(aug_img)
        axes[row_idx, col_idx].set_title(f'Augmented #{col_idx}', fontsize=10)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.show()

## Step 9: Build Model

In [ ]:
def swish(x):
    return x * tf.sigmoid(x)

def mish(x):
    return x * tf.tanh(tf.nn.softplus(x))

def build_model(input_shape, num_classes):
    base_model = EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    base_model.trainable = False  # Freeze initially

    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, kernel_regularizer=l2(1e-4))(x)
    x = layers.Lambda(swish)(x)
    x = Dropout(0.4)(x)
    x = Dense(256, kernel_regularizer=l2(1e-4))(x)
    x = layers.Lambda(mish)(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    return Model(inputs=inputs, outputs=outputs, name='SkinCancer_EfficientNetB3'), base_model

model, base_model = build_model(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=NUM_CLASSES
)

model.summary()

## Step 10: Phase 1 — Train Head Only

In [ ]:
BATCH_SIZE = 32
import warnings
warnings.filterwarnings('ignore')

# Phase 1 callbacks
callbacks_phase1 = [
    EarlyStopping(
        monitor='val_auc',
        patience=10,
        restore_best_weights=True,
        mode='max'
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

model.compile(
    optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)]
)

print('Phase 1: Training head with frozen base...')
history1 = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=20,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
    verbose=1
)

print(f'Phase 1 complete. Best val_auc: {max(history1.history["val_auc"]):.4f}')
print(f'Phase 1 complete. Best accuracy: {max(history1.history["accuracy"]):.4f}')

## Step 11: Phase 2 — Fine-Tune Top Layers

In [ ]:
base_model.trainable = True
total_layers = len(base_model.layers)
fine_tune_from = int(total_layers * 0.60)  # Unfreeze top 40%

for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

for layer in base_model.layers[fine_tune_from:]:
    layer.trainable = True

# Freeze all BatchNorm layers
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print(f'Fine-tuning top {total_layers - fine_tune_from} / {total_layers} layers')

model.compile(
    optimizer=AdamW(learning_rate=5e-5, weight_decay=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)]
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_auc',
        patience=25,
        restore_best_weights=True,
        mode='max'
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_skin_cancer_model.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max'
    )
]

print('Phase 2: Fine-tuning...')
history2 = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=30,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
    verbose=1
)
print(f'Phase 2 complete. Best val_auc: {max(history2.history["val_auc"]):.4f}')
print(f'Phase 2 complete. Best accuracy: {max(history2.history["accuracy"]):.4f}')

## Step 12: Training History

In [ ]:
# Combine both phases
combined_acc = history1.history['accuracy'] + history2.history['accuracy']
combined_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
combined_loss = history1.history['loss'] + history2.history['loss']
combined_val_loss = history1.history['val_loss'] + history2.history['val_loss']
combined_auc = history1.history['auc'] + history2.history['auc']
combined_val_auc = history1.history['val_auc'] + history2.history['val_auc']
phase1_end = len(history1.history['accuracy'])

total_epochs = len(combined_acc)
print(f'Total epochs trained: {total_epochs} (Phase 1: {phase1_end}, Phase 2: {total_epochs - phase1_end})')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, train, val, title, ylabel in zip(
    axes,
    [combined_acc, combined_loss, combined_auc],
    [combined_val_acc, combined_val_loss, combined_val_auc],
    ['Accuracy', 'Loss', 'AUC-ROC'],
    ['Accuracy', 'Loss', 'AUC']
):
    ax.plot(train, label='Train')
    ax.plot(val, label='Val')
    ax.axvline(x=phase1_end, color='gray', linestyle='--', alpha=0.7, label='Phase 2 start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 13: Validation Performance

In [ ]:
y_val_probs = model.predict(X_val)
y_val_pred  = np.argmax(y_val_probs, axis=1)

acc  = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
rec  = recall_score(y_val, y_val_pred, average='weighted', zero_division=0)
f1   = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
y_val_bin = label_binarize(y_val, classes=list(range(NUM_CLASSES)))
auc = roc_auc_score(y_val_bin, y_val_probs, multi_class='ovr', average='weighted')

print('=' * 45)
print('     VALIDATION PERFORMANCE (KPIs)')
print('=' * 45)
print(f'Accuracy (weighted):  {acc:.4f}')
print(f'Precision (weighted): {prec:.4f}')
print(f'Recall (weighted):    {rec:.4f}')
print(f'F1-Score (weighted):  {f1:.4f}')
print(f'AUC-ROC (weighted):   {auc:.4f}')
print('=' * 45)
print()


## Step 14: Confusion Matrix (Validation)

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.savefig('confusion_matrix_val.png', dpi=150, bbox_inches='tight')
plt.show()

### Validation Set — Sample Predictions with Images

In [ ]:

risk_levels = {
    'mel':   (' High Risk',    'red'),
    'bcc':   (' High Risk',    'red'),
    'akiec': (' Moderate Risk','darkorange'),
    'bkl':   (' Low Risk',     'goldenrod'),
    'df':    (' Benign',       'green'),
    'nv':    (' Benign',       'green'),
    'vasc':  (' Low Risk',     'goldenrod'),
}

sample_indices_val = np.random.RandomState(99).choice(len(X_val), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(sample_indices_val):
    img_display = reverse_preprocess(X_val[idx])
    axes[i].imshow(img_display)

    true_label = classes[y_val[idx]]
    pred_label = classes[y_val_pred[idx]]
    conf = y_val_probs[idx][y_val_pred[idx]]

    correct = true_label == pred_label
    title_color = 'green' if correct else 'red'

    risk_text, risk_color = risk_levels.get(pred_label, ('Unknown', 'gray'))

    axes[i].set_title(
        f'True: {true_label}  |  Pred: {pred_label} ({conf:.2f})\n'
        f'Risk: {risk_text}',
        color=title_color, fontsize=8.5
    )

    for spine in axes[i].spines.values():
        spine.set_edgecolor(risk_color)
        spine.set_linewidth(3)
        spine.set_visible(True)

    axes[i].axis('off')

plt.suptitle(
    'Validation Sample Predictions\n'
    'Green Title=Correct | Red Title=Incorrect | Border=Risk Level',
    fontsize=13
)
plt.tight_layout()
plt.savefig('val_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 15: Final Test Performance

In [ ]:
print('Loading best saved model...')
best_model = tf.keras.models.load_model(
    'best_skin_cancer_model.keras',
    custom_objects={'swish': swish, 'mish': mish}
)

y_test_probs = best_model.predict(X_test)
y_test_pred  = np.argmax(y_test_probs, axis=1)

test_acc  = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_rec  = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1   = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
y_test_bin = label_binarize(y_test, classes=list(range(NUM_CLASSES)))
test_auc   = roc_auc_score(y_test_bin, y_test_probs, multi_class='ovr', average='weighted')

print('\n' + '=' * 50)
print('     FINAL TEST PERFORMANCE (KPIs)')
print('     (Held-Out Test Set — Never Seen During Training)')
print('=' * 50)
print(f'Accuracy (weighted):  {test_acc:.4f}')
print(f'Precision (weighted): {test_prec:.4f}')
print(f'Recall (weighted):    {test_rec:.4f}')
print(f'F1-Score (weighted):  {test_f1:.4f}')
print(f'AUC-ROC (weighted):   {test_auc:.4f}')
print('=' * 50)
print()

## Step 16: Test Set Confusion Matrix & Sample Predictions

In [ ]:
# @title
cm_test = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=classes)
disp_test.plot(cmap='Oranges', xticks_rotation=45)
plt.title('Confusion Matrix — Final Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix_test.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
sample_indices = np.random.choice(len(X_test), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    img_display = reverse_preprocess(X_test[idx])
    axes[i].imshow(img_display)

    true_label = classes[y_test[idx]]
    pred_label = classes[y_test_pred[idx]]
    conf = y_test_probs[idx][y_test_pred[idx]]

    title_color = 'green' if true_label == pred_label else 'red'
    risk_text, risk_color = risk_levels.get(pred_label, ('Unknown', 'gray'))

    axes[i].set_title(
        f'True: {true_label}  |  Pred: {pred_label} ({conf:.2f})\n'
        f'Risk: {risk_text}',
        color=title_color, fontsize=8.5
    )

    for spine in axes[i].spines.values():
        spine.set_edgecolor(risk_color)
        spine.set_linewidth(3)
        spine.set_visible(True)

    axes[i].axis('off')

plt.suptitle(
    'Test Sample Predictions\n'
    'Green Title=Correct | Red Title=Incorrect | Border=Risk Level',
    fontsize=13
)
plt.tight_layout()
plt.savefig('test_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 17: Per-Class Accuracy Visualization

In [ ]:
# Per-class accuracy bar chart
per_class_acc = cm_test.diagonal() / cm_test.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(classes, per_class_acc * 100, color=plt.cm.viridis(np.linspace(0.3, 0.9, NUM_CLASSES)))
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_xlabel('Skin Lesion Class', fontsize=12)
ax.set_title('Per-Class Test Accuracy', fontsize=14)
ax.set_ylim(0, 100)
for bar, acc_val in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{acc_val*100:.1f}%', ha='center', va='bottom', fontsize=10)
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 18: Saving the Best Weighted Model

In [ ]:
# Save model
model.save('skin_cancer_model.keras')
print('Model saved!')

# Save class labels
import json
class_names = list(le.classes_)
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print('Class names saved:', class_names)

Model saved!
Class names saved: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


## Step 19: Testing on Local Image

In [ ]:
model = load_model(
    'best_skin_cancer_model.keras',
    custom_objects={'swish': tf.keras.activations.swish}
)

class_names = list(le.classes_)

IMAGE_PATHS = [
    'data/test-image1.jpg',
    'data/test-image2.jpg',
]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for i, image_path in enumerate(IMAGE_PATHS):
    img = Image.open(image_path).convert('RGB')
    img_array = preprocess_input(np.array(img.resize((224, 224))).astype(np.float32))
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    top_idx = np.argmax(preds)

    pred_class = class_names[top_idx]
    risk_text, risk_color = risk_levels.get(pred_class, ('Unknown', 'gray'))

    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(
        f'Predicted: {pred_class} \nRisk: {risk_text}',
        fontsize=12, color=risk_color, fontweight='bold'
    )
    for spine in axes[i].spines.values():
        spine.set_edgecolor(risk_color)
        spine.set_linewidth(4)
        spine.set_visible(True)

plt.suptitle('Skin Lesion Predictions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('local_image_prediction.png', dpi=150, bbox_inches='tight')
plt.show()